# Robotiq Gripper Tutorial

This notebook shows how to control a Robotiq gripper in RobotBlockSet.

The RobotBlockset wrapper inherits from `pyrobotiqgripper.RobotiqGripper` and keeps the high-level RobotBlockset API in **meters**. Internally, the backend communicates with the gripper over Modbus RTU and can use either bit-based commands or millimeter-based commands after calibration.

Relevant references:

- `robotiq_gripper(portname="...", slave_address=9, activate=True, calibrate=False, width_min=0.0, width_max=0.085)` creates the gripper interface and connects immediately.
- The upstream `pyrobotiqgripper` API documents `connect()`, `activate()`, `move()`, `open()`, `close()`, `move_mm()`, `readStatus()`, `getPosition()`, `getPositionmm()`, and `calibrate()`.
- According to the upstream documentation, activation makes the gripper perform a full opening/closing cycle, so the fingers must be free to move.


## 1. Import the wrapper

Use the RobotBlockset wrapper when you want a gripper object that fits the rest of the RobotBlockset interface. The wrapper exposes convenience methods such as `Open()`, `Close()`, `Move()`, `Grasp()`, `GetWidth()`, and `GetState()`.

In [1]:
from robotblockset.robotiq.grippers_robotiq import robotiq_gripper

## 2. Create and connect the gripper

The wrapper calls the upstream constructor with `com_port=portname` and `device_id=slave_address`, then runs `connect()` automatically.

- `portname="COM4"` is an example for Windows. Replace it with the serial port used by your adapter.
- `portname="auto"` uses the backend auto-detection logic.
- `activate=True` activates the gripper during construction.
- `calibrate=False` keeps the setup simple; calibration can be done later when you want explicit mm support.

For most Robotiq 2F setups the Modbus slave address is `9`.

In [2]:
g = robotiq_gripper(portname="COM4")

Activation completed. Activation time :  0.0


## 3. Move to a known reference pose

`Homing()` in the RobotBlockset wrapper opens the gripper to the configured maximum width. This is a good first command after connecting because it leaves the fingers in a known open state.

If you need to force a new activation cycle, `Homing(reactivate=True)` first calls `Activate()` and then opens the gripper again.

In [3]:
g.Homing()

True

## 4. Basic open and close commands

Use `Close()` to move to `width_min` and `Open()` to move to `width_max`. Both methods accept optional `speed` and `force` arguments in the 0..255 range, matching the upstream Robotiq interface.

In [4]:
g.Close()

True

In [5]:
g.Open()

True

## 5. Move to a specific opening width

`Move(width, speed=255, force=255)` is the main RobotBlockset command for setting the opening.

- `width` is expressed in **meters**.
- The wrapper clamps the request to `[width_min, width_max]`.
- If the backend has been calibrated, the wrapper forwards the command through `move_mm()`.
- Otherwise it converts the requested width to the Robotiq 0..255 position scale and uses the upstream `move()` call.

For a standard 85 mm gripper, `0.04` means a 40 mm opening.

In [14]:
g.Move(0.04)

True

## 6. Read the current width and state

The `width` property calls `GetWidth()`, which refreshes the backend status and returns the current opening in meters. `GetState()` returns the RobotBlockset state derived from the measured width and grasp detection.

In [7]:
{
    "width_m": g.width,
    "state": g.GetState(),
}

{'width_m': 0.049666666666666665, 'state': 'Undefined'}

## 7. Perform a grasp

`Grasp(width=..., speed=..., force=...)` first performs a move and then checks the status registers to estimate whether contact was detected. In this wrapper, contact is reported when the Robotiq status flag `gOBJ == 2`.

This is useful when you want one command that both closes toward a target width and updates the internal RobotBlockset state to a grasped condition.

In [8]:
g.Grasp(width=0.04, speed=255, force=10)

True

## 8. Optional calibration for mm-based control

The upstream API documents that `move_mm()` and `getPositionmm()` require calibration. The RobotBlockset wrapper exposes this through `Calibrate(width_min=..., width_max=...)`, where the values are still given in **meters**.

For a typical 2F-85 gripper, a common calibration is 0 mm closed and 85 mm open, which corresponds to `0.0` and `0.085` meters.

In [13]:
# Optional: run once if you want explicit mm-based backend support
g.Calibrate(width_min=0.0, width_max=0.085)

## 9. Clean shutdown

When you finish the session, disconnect the serial connection cleanly.

In [12]:
# g.disconnect()